In [472]:
import pandas as pd
import numpy as np
import random
from collections import Counter

# PROLOGUE PARAMETER

In [473]:
POPULASI = 50

In [474]:
guru_df = pd.read_csv('../dataset/guru.csv')
kelas_df = pd.read_csv('../dataset/kelas.csv')
mapel_df = pd.read_csv('../dataset/mapel.csv')
relasi_guru_mapel_df = pd.read_csv('../dataset/relasi_guru_mapel.csv')
slot_df = pd.read_csv('../dataset/slot.csv')

# 1. join relasi

In [475]:
# guru_df = relasi_guru_mapel_df = mapel-df
join = (
    relasi_guru_mapel_df
    .merge(guru_df, on="guru_id", how="left")
    .merge(mapel_df, on="mapel_id", how="left")
)

# penambahan kolom "total"
join["total"] = (
    join.groupby("guru_id")["durasi"]
    .transform("sum")
) 
# join.head(3)

In [476]:
# join.count()

# 2. Menggunakan Numerik

### konversi hari ke numerik

In [477]:
# dictionary hari -> numerik
mapping_hari = {
    "Senin": 1,
    "Selasa": 2,
    "Rabu": 3,
    "Kamis": 4,
    "Jumat": 5,
}

### mengambil kolom penting

In [478]:
# mengambil kolom numerik
relasi = join[[
    "guru_id",
    "mapel_id",
    "jam_per_minggu",
    "tingkatan",
    "durasi",
    "total",
    "MGMP"
]].copy()

# melakukan konversi hari ke numerik pakai mapping
relasi["MGMP"] = relasi["MGMP"].map(mapping_hari)
# relasi.info()

### Dict Mapel dan jam

In [479]:
mapel_jam = dict(
    zip(mapel_df["mapel_id"], mapel_df["jam_per_minggu"])
)
# mapel_jam

### mapping hari dan jumlah slot

In [480]:
# mapping jumlah slot per hari
slot_per_hari = {
    1 : 8, # senin
    2 : 8, # Selasa
    3 : 8, # rabu
    4 : 7, # kamis
    5 : 5, # jumat
}

### count kelas

In [481]:
# mengambil total kelas (27)
total_kelas = kelas_df["kelas_id"].count()
# total_kelas

### Batas siang
digunakan untuk mapel PJOK (8)

In [482]:
batas_siang = {
    0: 5,  # Senin  → index 6,7 dilarang
    1: 5,  # Selasa → index 6,7 dilarang
    2: 4,  # Rabu   → index 5,6,7 dilarang
    3: 5,  # Kamis  → index 6 dilarang
    4: 4   # Jumat  → index 5 dilarang
}

# 3. Groupping guru  
mapel_id dan tingkatan

In [483]:
guru_by_mapel = (
    relasi
    .groupby(["mapel_id", "tingkatan"])["guru_id"]
    .apply(list)
    .to_dict()
)
# guru_by_mapel

# 4. Daftar Variabel

variabel variabel yang telah diproses dan siap digunakan
| Nama Variabel| Keterangan Singkat |
|--------------|-------------------|
| `guru_df` |  Original guru |
| `kelas_df` |  Original kelas |
| `mapel_df` |  Original mapel |
| `relasi_guru_mapel_df` |  Original relasi |
| `slot_df` |  Original slot |
| `relasi` |  Relasi guru & mapel (93 rows) |
| `guru_by_mapel` |  Guru groupping |
| `total_kelas`| 27 (int)|
|`slot_per_hari`| dictionary |
| `mapel_jam`| dictionary|
| `batas_siang`| dictionary |


# 5. Fungsi Inisialisasi Individu

In [484]:
def individuConstruct(slot_per_hari, total_kelas, guru_by_mapel, relasi):

    mapel_id = list(range(1, 14))     # 13 mapel
    tingkatan = relasi["tingkatan"].iloc[0]

    # preprocess guru per mapel
    guru_mapel_list = {
        m: guru_by_mapel.get((m, tingkatan), [])
        for m in mapel_id
    }

    individu = []

    for _ in range(total_kelas):

        kelas = []

        # ===== GEN MAPEL PER HARI =====
        for hari in sorted(slot_per_hari.keys()):  # 1..5
            jumlah_slot = slot_per_hari[hari]
            gen_hari = [random.choice(mapel_id) for _ in range(jumlah_slot)]
            kelas.append(gen_hari)

        # ===== GEN GURU (13 MAPEL) =====
        gen_guru = [
            random.choice(guru_mapel_list[m]) if guru_mapel_list[m] else 0
            for m in mapel_id
        ]

        kelas.append(gen_guru)

        individu.append(kelas)

    return individu


In [485]:
individu = individuConstruct(slot_per_hari, total_kelas, guru_by_mapel, relasi)
# individu

In [486]:
# individu

In [487]:
# individu[0]

In [488]:
populasi = []
for i in range(POPULASI):
    individu = individuConstruct(slot_per_hari, total_kelas, guru_by_mapel, relasi)
    populasi.append(individu)

In [489]:
# populasi

In [490]:
# individu =   
# [ # ini 1 individu  
#         [ # ini 1 kelas  
#                 [m1, m2, m3, m4, m5, m6, m7, m8],   # Senin (8)  
#                 [m1, m2, m3, m4, m5, m6, m7, m8],   # Selasa (8)  
#                 [m1, m2, m3, m4, m5, m6, m7, m8],   # Rabu (8)  
#                 [m1, m2, m3, m4, m5, m6, m7],       # Kamis (7)  
#                 [m1, m2, m3, m4, m5],                # Jumat (5)  
#                 [g1, g2, g3, g4, g5, sampai g13]        # guru pengajar (13)  
#         ],  
#         [  
#                 # kelas lain  
#         ]  
# ]  

# 6. Evaluasi Individu

### 6.1. Guru hanya boleh mengajar 1 slot waktu, tidak boleh lebih

In [491]:
def guru_bentrok(individu, violation_cost):
    pelanggaran = 0
    cost = 0

    slot_harian = individu[0][:-1]

    for hari_id, hari in enumerate(slot_harian):
        for jam_id in range(len(hari)):
            guru_used = set()

            for kelas in individu:
                mapel = kelas[hari_id][jam_id]

                if mapel == 0:
                    continue

                guru = kelas[-1][mapel - 1]

                if guru in guru_used:
                    pelanggaran += 1
                    cost += violation_cost
                else:
                    guru_used.add(guru)

    return pelanggaran, cost

In [492]:
for individu in populasi:
    pelanggaran, cost = guru_bentrok(individu, 100)
    print("Hard Constraint: ", pelanggaran)
    print("Cost: ", cost)

Hard Constraint:  381
Cost:  38100
Hard Constraint:  374
Cost:  37400
Hard Constraint:  374
Cost:  37400
Hard Constraint:  368
Cost:  36800
Hard Constraint:  374
Cost:  37400
Hard Constraint:  364
Cost:  36400
Hard Constraint:  366
Cost:  36600
Hard Constraint:  363
Cost:  36300
Hard Constraint:  364
Cost:  36400
Hard Constraint:  362
Cost:  36200
Hard Constraint:  379
Cost:  37900
Hard Constraint:  383
Cost:  38300
Hard Constraint:  370
Cost:  37000
Hard Constraint:  368
Cost:  36800
Hard Constraint:  373
Cost:  37300
Hard Constraint:  362
Cost:  36200
Hard Constraint:  392
Cost:  39200
Hard Constraint:  365
Cost:  36500
Hard Constraint:  382
Cost:  38200
Hard Constraint:  362
Cost:  36200
Hard Constraint:  377
Cost:  37700
Hard Constraint:  362
Cost:  36200
Hard Constraint:  357
Cost:  35700
Hard Constraint:  353
Cost:  35300
Hard Constraint:  370
Cost:  37000
Hard Constraint:  377
Cost:  37700
Hard Constraint:  354
Cost:  35400
Hard Constraint:  370
Cost:  37000
Hard Constraint:  38

### 6.2. konfigurasi mapel  
2 jam -> 1 hari  
3 jam -> 1 hari    
4 jam -> 2 hari berbeda [2,2]  
5 jam -> 2 hari berbeda [2,3]

In [493]:
def konfigurasi_mapel(individu, mapel_jam, violation_cost):
    pelanggaran = 0
    cost = 0

    for kelas in individu:
        slot_harian = kelas[:-1]  # senin–jumat

        mapel_hari = {}

        for hari_id, hari in enumerate(slot_harian):
            for slot_id, mapel in enumerate(hari):
                if mapel not in mapel_hari:
                    mapel_hari[mapel] = {}
                if hari_id not in mapel_hari[mapel]:
                    mapel_hari[mapel][hari_id] = []
                mapel_hari[mapel][hari_id].append(slot_id)

        # evaluasi per mapel
        for mapel_id, distribusi_hari in mapel_hari.items():
            total_jam = mapel_jam.get(mapel_id, 0)

            # rule jumlah hari
            hari_terpakai = [len(v) for v in distribusi_hari.values()]

            if total_jam in (2, 3):
                # mapel dengan 2 , 3 jam per minggu
                if len(distribusi_hari) != 1:
                    pelanggaran += 1
                    cost += violation_cost

            elif total_jam == 4:
                # mapel dengan 4 jam per minggu
                if sorted(hari_terpakai) != [2, 2]:
                    pelanggaran += 1
                    cost += violation_cost

            elif total_jam == 5:
                # mapel dengan 5 jam per minggu
                if sorted(hari_terpakai) != [2, 3]:
                    pelanggaran += 1
                    cost += violation_cost

            # mapel harus pada slot yang berdekatan pada hari yang sama
            for slot_list in distribusi_hari.values():
                if len(slot_list) > 1:
                    slot_list = sorted(slot_list)
                    for i in range(len(slot_list) - 1):
                        if slot_list[i + 1] - slot_list[i] != 1:
                            pelanggaran += 1
                            cost += violation_cost
                            break

    return pelanggaran, cost

In [494]:
for individu in populasi:
    pelanggaran, cost = konfigurasi_mapel(individu,mapel_jam, 100)
    print("Hard Constraint: ", pelanggaran)
    print("Cost: ", cost)

Hard Constraint:  386
Cost:  38600
Hard Constraint:  415
Cost:  41500
Hard Constraint:  394
Cost:  39400
Hard Constraint:  403
Cost:  40300
Hard Constraint:  395
Cost:  39500
Hard Constraint:  417
Cost:  41700
Hard Constraint:  401
Cost:  40100
Hard Constraint:  414
Cost:  41400
Hard Constraint:  391
Cost:  39100
Hard Constraint:  408
Cost:  40800
Hard Constraint:  415
Cost:  41500
Hard Constraint:  401
Cost:  40100
Hard Constraint:  389
Cost:  38900
Hard Constraint:  397
Cost:  39700
Hard Constraint:  414
Cost:  41400
Hard Constraint:  399
Cost:  39900
Hard Constraint:  418
Cost:  41800
Hard Constraint:  408
Cost:  40800
Hard Constraint:  420
Cost:  42000
Hard Constraint:  399
Cost:  39900
Hard Constraint:  396
Cost:  39600
Hard Constraint:  394
Cost:  39400
Hard Constraint:  417
Cost:  41700
Hard Constraint:  407
Cost:  40700
Hard Constraint:  398
Cost:  39800
Hard Constraint:  402
Cost:  40200
Hard Constraint:  398
Cost:  39800
Hard Constraint:  421
Cost:  42100
Hard Constraint:  40

### 6.3. Mapel PJOK

In [495]:
def mapel_pjok(individu, batas_siang, violation_cost):
    pelanggaran = 0
    cost = 0

    for kelas in individu:
        slot_harian = kelas[:-1] # hari saja

        for hari_id, hari in enumerate(slot_harian):
            batas = batas_siang[hari_id]

            for slot_id, mapel in enumerate(hari):
                if mapel == 8 and slot_id > batas:
                    pelanggaran += 1
                    cost += violation_cost
                    
    return pelanggaran, cost

In [496]:
for individu in populasi:
    pelanggaran, cost = mapel_pjok(individu,batas_siang, 100)
    print("Hard Constraint: ", pelanggaran)
    print("Cost: ", cost)

Hard Constraint:  14
Cost:  1400
Hard Constraint:  12
Cost:  1200
Hard Constraint:  18
Cost:  1800
Hard Constraint:  13
Cost:  1300
Hard Constraint:  20
Cost:  2000
Hard Constraint:  13
Cost:  1300
Hard Constraint:  11
Cost:  1100
Hard Constraint:  19
Cost:  1900
Hard Constraint:  22
Cost:  2200
Hard Constraint:  13
Cost:  1300
Hard Constraint:  15
Cost:  1500
Hard Constraint:  15
Cost:  1500
Hard Constraint:  15
Cost:  1500
Hard Constraint:  12
Cost:  1200
Hard Constraint:  14
Cost:  1400
Hard Constraint:  24
Cost:  2400
Hard Constraint:  15
Cost:  1500
Hard Constraint:  20
Cost:  2000
Hard Constraint:  17
Cost:  1700
Hard Constraint:  14
Cost:  1400
Hard Constraint:  21
Cost:  2100
Hard Constraint:  16
Cost:  1600
Hard Constraint:  21
Cost:  2100
Hard Constraint:  19
Cost:  1900
Hard Constraint:  14
Cost:  1400
Hard Constraint:  25
Cost:  2500
Hard Constraint:  16
Cost:  1600
Hard Constraint:  15
Cost:  1500
Hard Constraint:  17
Cost:  1700
Hard Constraint:  15
Cost:  1500
Hard Const

### 6.4. Durasi mengajar Guru

In [497]:
def durasi_guru(individu, relasi, violation_cost):
    pelanggaran = 0
    cost = 0

    # lookup (guru, mapel) -> durasi
    durasi_lookup = {
        (row.guru_id, row.mapel_id) : row.durasi
        for row in relasi.itertuples(index=False)
    }

    for kelas in individu:
        slot_harian = kelas[:-1]
        guru_mapel = kelas[-1]

    # menghitung total jam
    jam_guru_aktual = Counter()

    for hari in slot_harian:
        for mapel in hari:
            guru = guru_mapel[mapel - 1]
            jam_guru_aktual[(guru, mapel)] += 1

    for (guru, mapel), jam_aktual in jam_guru_aktual.items():
        durasi_wajib = durasi_lookup.get((guru, mapel))

        if durasi_wajib is None:
            # tidak valid di relasi.df
            pelanggaran += 1
            cost += violation_cost

        elif jam_aktual != durasi_wajib:
            pelanggaran += abs(jam_aktual - durasi_wajib)
            cost += violation_cost * abs(jam_aktual - durasi_wajib)

    return pelanggaran, cost

In [498]:
for individu in populasi:
    pelanggaran, cost = durasi_guru(individu, relasi, 100)
    print("Hard Constraint: ", pelanggaran)
    print("Cost: ", cost)

Hard Constraint:  135
Cost:  13500
Hard Constraint:  88
Cost:  8800
Hard Constraint:  126
Cost:  12600
Hard Constraint:  141
Cost:  14100
Hard Constraint:  67
Cost:  6700
Hard Constraint:  81
Cost:  8100
Hard Constraint:  126
Cost:  12600
Hard Constraint:  135
Cost:  13500
Hard Constraint:  110
Cost:  11000
Hard Constraint:  138
Cost:  13800
Hard Constraint:  101
Cost:  10100
Hard Constraint:  136
Cost:  13600
Hard Constraint:  134
Cost:  13400
Hard Constraint:  137
Cost:  13700
Hard Constraint:  105
Cost:  10500
Hard Constraint:  75
Cost:  7500
Hard Constraint:  94
Cost:  9400
Hard Constraint:  124
Cost:  12400
Hard Constraint:  121
Cost:  12100
Hard Constraint:  111
Cost:  11100
Hard Constraint:  145
Cost:  14500
Hard Constraint:  79
Cost:  7900
Hard Constraint:  123
Cost:  12300
Hard Constraint:  104
Cost:  10400
Hard Constraint:  116
Cost:  11600
Hard Constraint:  117
Cost:  11700
Hard Constraint:  121
Cost:  12100
Hard Constraint:  124
Cost:  12400
Hard Constraint:  134
Cost:  134

### 6.5. Mapel sesuai dengan jam_per_minggu

In [499]:
def mapel_jam_per_minggu(individu, mapel_jam, violation_cost):
    pelanggaran = 0
    cost = 0

    for kelas in individu:
        slot_harian = kelas[:-1]

        semua_slot = []
        for hari in slot_harian:
            semua_slot.extend(hari)

        hitung_mapel = Counter(semua_slot)

        for mapel_id, jam_wajib in mapel_jam.items():
            jam_aktual = hitung_mapel.get(mapel_id, 0)

            if jam_aktual != jam_wajib:
                diff = abs(jam_aktual - jam_wajib)

                pelanggaran += diff
                cost += diff * violation_cost
                
    return pelanggaran, cost

In [500]:
for individu in populasi:
    pelanggaran, cost = mapel_jam_per_minggu(individu, mapel_jam, 100)
    print("Hard Constraint: ", pelanggaran)
    print("Cost: ", cost)

Hard Constraint:  494
Cost:  49400
Hard Constraint:  514
Cost:  51400
Hard Constraint:  542
Cost:  54200
Hard Constraint:  492
Cost:  49200
Hard Constraint:  512
Cost:  51200
Hard Constraint:  520
Cost:  52000
Hard Constraint:  486
Cost:  48600
Hard Constraint:  510
Cost:  51000
Hard Constraint:  486
Cost:  48600
Hard Constraint:  508
Cost:  50800
Hard Constraint:  548
Cost:  54800
Hard Constraint:  530
Cost:  53000
Hard Constraint:  516
Cost:  51600
Hard Constraint:  514
Cost:  51400
Hard Constraint:  476
Cost:  47600
Hard Constraint:  496
Cost:  49600
Hard Constraint:  532
Cost:  53200
Hard Constraint:  514
Cost:  51400
Hard Constraint:  538
Cost:  53800
Hard Constraint:  482
Cost:  48200
Hard Constraint:  544
Cost:  54400
Hard Constraint:  552
Cost:  55200
Hard Constraint:  510
Cost:  51000
Hard Constraint:  512
Cost:  51200
Hard Constraint:  542
Cost:  54200
Hard Constraint:  528
Cost:  52800
Hard Constraint:  538
Cost:  53800
Hard Constraint:  500
Cost:  50000
Hard Constraint:  49